In [ ]:
%pip install pandas 


In [ ]:
import pandas as pd
print(pd.__version__)

In [ ]:
import pandas as pd

df = pd.read_csv("../data/bank_defaulter_raw_anomalous.csv")

print("Shape:", df.shape)
df.head()

In [ ]:
df["default"].value_counts()

In [ ]:
df["default"].value_counts(normalize=True) * 100

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

missing = missing[missing["missing_count"] > 0]

missing.sort_values("missing_percentage", ascending=False)

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())

In [ ]:
df["employment_type"].value_counts(dropna=False)

In [ ]:
print("Rows before:", len(df))
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())

In [ ]:
duplicates = df[df["customer_id"].duplicated(keep=False)] \
    .sort_values("customer_id")

duplicates.head(20)


In [ ]:
df = df.drop_duplicates()

print("rows before:", len(df))

In [ ]:
print("Duplicate rows remaining:", df.duplicated().sum())


In [ ]:
df.dtypes

In [ ]:
numeric_columns = [
    "age",
    "annual_income",
    "credit_score",
    "loan_amount",
    "loan_tenure_months",
    "existing_loans",
    "monthly_debt",
    "monthly_expenses",
    "employment_years",
    "bank_account_age_years",
    "num_credit_cards",
    "missed_payments_last_year",
    "previous_defaults",
    "credit_utilization",
    "debt_to_income_ratio",
    "savings_balance",
    "transaction_frequency",
    "loan_to_income_ratio",
    "default"
]

for col in numeric_columns:
    print(f"{col:30} -> {df[col].dtype}")

In [ ]:
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df.dtypes

In [ ]:
df[numeric_columns].isna().sum()

In [ ]:
missing_columns = [
    "annual_income",
    "credit_score",
    "monthly_debt",
    "employment_years",
    "savings_balance"
]

df[missing_columns].describe().T

In [ ]:
df[missing_columns].skew()

In [ ]:
for col in missing_columns:
    print(f"{col}: {df[col].median():.2f}")

In [ ]:
for col in missing_columns:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
df[missing_columns].isna().sum()


In [ ]:
invalid_age = df[
    (df["age"] < 18) | (df["age"] > 100)
]

print("Invalid age records:", len(invalid_age))
invalid_age[["customer_id", "age"]].head(20)

In [ ]:
invalid_credit = df[
    (df["credit_score"] < 300) | (df["credit_score"] > 850)
]

print("Invalid credit-score records:", len(invalid_credit))
invalid_credit[["customer_id", "credit_score"]].head(20)

In [ ]:
invalid_loan = df[
    df["loan_amount"] <= 0
]

print("Invalid loan records:", len(invalid_loan))
invalid_loan[["customer_id", "loan_amount"]].head(20)

In [ ]:
invalid_employment = df[
    (df["employment_years"] < 0) |
    (df["employment_years"] > df["age"] - 18)
]

print("Invalid employment records:", len(invalid_employment))

invalid_employment[
    ["customer_id", "age", "employment_years"]
].head(20)

In [ ]:
import numpy as np

# Invalid ages
df.loc[(df["age"] < 18) | (df["age"] > 100), "age"] = np.nan

# Invalid credit scores
df.loc[
    (df["credit_score"] < 300) | (df["credit_score"] > 850),
    "credit_score"
] = np.nan

# Invalid loan amounts
df.loc[df["loan_amount"] <= 0, "loan_amount"] = np.nan

# Invalid employment experience
df.loc[
    (df["employment_years"] < 0) |
    (df["employment_years"] > df["age"] - 18),
    "employment_years"
] = np.nan

In [ ]:
check_columns = [
    "age",
    "credit_score",
    "loan_amount",
    "employment_years"
]

df[check_columns].isna().sum()

In [ ]:
for col in check_columns:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
df[check_columns].isna().sum()

In [ ]:
print(
    df[
        df["employment_years"] > df["age"] - 18
    ][
        ["customer_id", "age", "employment_years"]
    ].head(20)
)

In [ ]:
max_employment_years = df["age"] - 18

invalid_employment = (
    df["employment_years"] > max_employment_years
)

print("Logically invalid employment records:",
      invalid_employment.sum())

In [ ]:
df.loc[
    invalid_employment,
    ["customer_id", "age", "employment_years"]
].head(20)

In [ ]:
df.loc[
    invalid_employment,
    "employment_years"
] = max_employment_years[invalid_employment]

In [ ]:
remaining_invalid = (
    df["employment_years"] > (df["age"] - 18)
).sum()

print("Remaining invalid employment records:",
      remaining_invalid)

In [ ]:
df["employment_type"].value_counts(dropna=False)

In [ ]:
df["employment_type"] = (
    df["employment_type"]
    .str.strip()
    .str.lower()
)

In [ ]:
df["employment_type"].value_counts()

In [ ]:
employment_mapping = {
    "salaried": "Salaried",
    "salary": "Salaried",

    "self-employed": "Self-Employed",
    "self employed": "Self-Employed",

    "business": "Business",
    "biz": "Business",

    "unemployed": "Unemployed",
    "no job": "Unemployed"
}

In [ ]:
df["employment_type"] = df["employment_type"].replace(
    employment_mapping
)

In [ ]:
df["employment_type"].value_counts(dropna=False)

In [ ]:
valid_categories = [
    "Salaried",
    "Self-Employed",
    "Business",
    "Unemployed"
]

unexpected = df[
    ~df["employment_type"].isin(valid_categories)
]

print("Unexpected employment categories:", len(unexpected))
unexpected[["customer_id", "employment_type"]].head(20)

In [ ]:
outlier_columns = [
    "annual_income",
    "loan_amount",
    "monthly_debt",
    "monthly_expenses",
    "savings_balance",
    "transaction_frequency"
]

for col in outlier_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) |
        (df[col] > upper)
    ]

    print(f"\n{col}")
    print(f"Q1: {Q1:.2f}")
    print(f"Q3: {Q3:.2f}")
    print(f"IQR: {IQR:.2f}")
    print(f"Lower bound: {lower:.2f}")
    print(f"Upper bound: {upper:.2f}")
    print(f"Outliers: {len(outliers)}")

In [ ]:
df[outlier_columns].describe().T

In [ ]:
for col in outlier_columns:
    print(f"\n===== {col} =====")
    print(df[col].nlargest(10).to_list())

In [ ]:
df[
    df["annual_income"] > 10_000_000
][[
    "customer_id",
    "age",
    "employment_type",
    "annual_income",
    "loan_amount",
    "credit_score",
    "default"
]].sort_values(
    "annual_income",
    ascending=False
).head(20)

In [ ]:
df[
    df["loan_amount"] > 10_000_000
][[
    "customer_id",
    "age",
    "employment_type",
    "annual_income",
    "loan_amount",
    "credit_score",
    "default"
]].sort_values(
    "loan_amount",
    ascending=False
).head(20)

In [ ]:
df.nlargest(
    20,
    "loan_to_income_ratio"
)[[
    "customer_id",
    "annual_income",
    "loan_amount",
    "loan_to_income_ratio",
    "credit_score",
    "default"
]]

In [ ]:
duplicate_ids = df[
    df["customer_id"].duplicated(keep=False)
].sort_values("customer_id")

print("Duplicate customer IDs:", duplicate_ids["customer_id"].nunique())
print("Rows involved:", len(duplicate_ids))

In [ ]:
duplicate_ids.head(10)


In [ ]:
duplicate_ids.groupby("customer_id").size().value_counts()

In [ ]:
df_before_duplicate_cleanup = df.copy()

In [ ]:
df = df.drop_duplicates(
    subset="customer_id",
    keep="first"
).reset_index(drop=True)

In [ ]:
print("Rows after duplicate-ID cleanup:", len(df))
print("Duplicate customer IDs remaining:",
      df["customer_id"].duplicated().sum())

In [ ]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

print("\nDuplicate rows:", df.duplicated().sum())

print("\nDuplicate customer IDs:",
      df["customer_id"].duplicated().sum())

In [ ]:
print("Invalid ages:",
      ((df["age"] < 18) | (df["age"] > 100)).sum())

print("Invalid credit scores:",
      ((df["credit_score"] < 300) |
       (df["credit_score"] > 850)).sum())

print("Invalid loan amounts:",
      (df["loan_amount"] <= 0).sum())

print("Invalid employment years:",
      (
          (df["employment_years"] < 0) |
          (df["employment_years"] > df["age"] - 18)
      ).sum())

In [ ]:
calculated_lti = (
    df["loan_amount"] / df["annual_income"]
)

difference = (
    calculated_lti - df["loan_to_income_ratio"]
).abs()

print("Maximum difference:", difference.max())
print("Rows with significant difference:", (difference > 0.01).sum())

In [ ]:
calculated_dti = (
    (df["monthly_debt"] + 
     df["loan_amount"] / df["loan_tenure_months"])
    /
    (df["annual_income"] / 12)
)

difference = (
    calculated_dti - df["debt_to_income_ratio"]
).abs()

print("Maximum difference:", difference.max())
print(
    "Rows with significant difference:",
    (difference > 0.01).sum()
)

In [ ]:
df["debt_to_income_ratio"].describe()

In [ ]:
df.nlargest(
    20,
    "debt_to_income_ratio"
)[[
    "customer_id",
    "annual_income",
    "loan_amount",
    "loan_tenure_months",
    "monthly_debt",
    "debt_to_income_ratio",
    "default"
]]

In [ ]:
df[
    df["annual_income"] > 10_000_000
][[
    "customer_id",
    "age",
    "employment_type",
    "annual_income",
    "loan_amount",
    "credit_score",
    "default"
]].sort_values(
    "annual_income",
    ascending=False
).head(20)

In [ ]:
df[
    df["loan_amount"] > 10_000_000
][[
    "customer_id",
    "age",
    "employment_type",
    "annual_income",
    "loan_amount",
    "credit_score",
    "default"
]].sort_values(
    "loan_amount",
    ascending=False
).head(20)

In [ ]:
calculated_lti = (
    df["loan_amount"] / df["annual_income"]
)

lti_difference = (
    calculated_lti - df["loan_to_income_ratio"]
).abs()

print(
    "Inconsistent LTI records:",
    (lti_difference > 0.01).sum()
)

In [ ]:
df.assign(
    calculated_lti=calculated_lti,
    lti_difference=lti_difference
).nlargest(
    20,
    "lti_difference"
)[[
    "customer_id",
    "annual_income",
    "loan_amount",
    "loan_to_income_ratio",
    "calculated_lti",
    "lti_difference",
    "credit_score",
    "default"
]]

In [ ]:
calculated_dti = (
    (df["monthly_debt"] +
     df["loan_amount"] / df["loan_tenure_months"])
    /
    (df["annual_income"] / 12)
)

dti_difference = (
    calculated_dti - df["debt_to_income_ratio"]
).abs()

print(
    "Inconsistent DTI records:",
    (dti_difference > 0.01).sum()
)

In [ ]:
df.assign(
    calculated_dti=calculated_dti,
    dti_difference=dti_difference
).nlargest(
    20,
    "dti_difference"
)[[
    "customer_id",
    "annual_income",
    "loan_amount",
    "monthly_debt",
    "loan_tenure_months",
    "debt_to_income_ratio",
    "calculated_dti",
    "dti_difference",
    "default"
]]

In [ ]:
df["loan_to_income_ratio"] = (
    df["loan_amount"] / df["annual_income"]
).round(3)

In [ ]:
df["debt_to_income_ratio"] = (
    (
        df["monthly_debt"] +
        df["loan_amount"] / df["loan_tenure_months"]
    )
    /
    (df["annual_income"] / 12)
).round(3)

In [ ]:
calculated_lti = (
    df["loan_amount"] / df["annual_income"]
)

calculated_dti = (
    (
        df["monthly_debt"] +
        df["loan_amount"] / df["loan_tenure_months"]
    )
    /
    (df["annual_income"] / 12)
)

print(
    "LTI mismatches:",
    (
        (calculated_lti - df["loan_to_income_ratio"]).abs() > 0.001
    ).sum()
)

print(
    "DTI mismatches:",
    (
        (calculated_dti - df["debt_to_income_ratio"]).abs() > 0.001
    ).sum()
)

In [ ]:
financial_columns = [
    "annual_income",
    "loan_amount",
    "monthly_debt",
    "monthly_expenses",
    "savings_balance",
    "transaction_frequency"
]

df[financial_columns].skew().sort_values(ascending=False)

In [ ]:
df[financial_columns].describe().T[
    ["min", "25%", "50%", "75%", "max"]
]

In [ ]:
extreme_columns = [
    "annual_income",
    "loan_amount",
    "monthly_debt",
    "monthly_expenses",
    "savings_balance",
    "transaction_frequency"
]

for col in extreme_columns:
    threshold = df[col].quantile(0.995)
    
    count = (df[col] > threshold).sum()
    
    print(
        f"{col}: threshold = {threshold:.2f}, "
        f"extreme records = {count}"
    )

In [ ]:
extreme_mask = (
    (df["annual_income"] > df["annual_income"].quantile(0.995)) |
    (df["loan_amount"] > df["loan_amount"].quantile(0.995)) |
    (df["monthly_debt"] > df["monthly_debt"].quantile(0.995)) |
    (df["monthly_expenses"] > df["monthly_expenses"].quantile(0.995)) |
    (df["savings_balance"] > df["savings_balance"].quantile(0.995)) |
    (df["transaction_frequency"] > df["transaction_frequency"].quantile(0.995))
)

extreme_records = df.loc[
    extreme_mask,
    [
        "customer_id",
        "age",
        "employment_type",
        "annual_income",
        "loan_amount",
        "monthly_debt",
        "monthly_expenses",
        "savings_balance",
        "transaction_frequency",
        "credit_score",
        "default"
    ]
]

print("Extreme records:", len(extreme_records))

extreme_records.head(30)

In [ ]:
df[
    df["transaction_frequency"] > 100
][[
    "customer_id",
    "annual_income",
    "transaction_frequency",
    "credit_score",
    "default"
]].sort_values(
    "transaction_frequency",
    ascending=False
)

In [ ]:
import numpy as np

df["log_income"] = np.log1p(df["annual_income"])
df["log_loan_amount"] = np.log1p(df["loan_amount"])
df["log_savings"] = np.log1p(df["savings_balance"])

In [ ]:
transaction_anomalies = df[
    df["transaction_frequency"] > 100
].sort_values(
    "transaction_frequency",
    ascending=False
)

print("Transaction anomalies:", len(transaction_anomalies))

transaction_anomalies[
    [
        "customer_id",
        "age",
        "employment_type",
        "annual_income",
        "transaction_frequency",
        "credit_score",
        "default"
    ]
]

In [ ]:
df.loc[
    df["transaction_frequency"] > 100,
    "transaction_frequency"
] = np.nan

In [ ]:
df["transaction_frequency"] = df[
    "transaction_frequency"
].fillna(
    df["transaction_frequency"].median()
)

In [ ]:
print(
    "Remaining transaction anomalies:",
    (df["transaction_frequency"] > 100).sum()
)

In [ ]:
print("========== FINAL DATA QUALITY CHECK ==========")

print("Shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

print("\nDuplicate rows:", df.duplicated().sum())

print(
    "Duplicate customer IDs:",
    df["customer_id"].duplicated().sum()
)

print(
    "\nInvalid ages:",
    ((df["age"] < 18) | (df["age"] > 100)).sum()
)

print(
    "Invalid credit scores:",
    ((df["credit_score"] < 300) |
     (df["credit_score"] > 850)).sum()
)

print(
    "Invalid loan amounts:",
    (df["loan_amount"] <= 0).sum()
)

print(
    "Invalid employment years:",
    (
        (df["employment_years"] < 0) |
        (df["employment_years"] > df["age"] - 18)
    ).sum()
)

print(
    "Transaction anomalies:",
    (df["transaction_frequency"] > 100).sum()
)

In [ ]:
df.to_csv(
    "../data/bank_defaulter_clean.csv",
    index=False
)

print("Clean dataset saved successfully!")

# Data Cleaning Summary

## 1. Missing Values
- Identified missing values in financial columns.
- Used median imputation because the affected variables were skewed.

## 2. Data Types
- Converted numerical columns to numeric format.
- Invalid numeric strings were converted to NaN.

## 3. Duplicates
- Removed exact duplicate rows.
- Removed duplicate customer IDs to maintain one record per customer.

## 4. Invalid Values
- Corrected impossible ages.
- Corrected invalid credit scores.
- Corrected non-positive loan amounts.
- Corrected logically impossible employment durations.

## 5. Categorical Data
- Standardized employment categories.

## 6. Derived Features
- Recalculated loan-to-income ratio.
- Recalculated debt-to-income ratio.

## 7. Extreme Values
- Investigated statistical outliers.
- Retained plausible extreme financial observations.
- Corrected deliberately corrupted transaction-frequency values.

## 8. Output
- Saved cleaned dataset as `bank_defaulter_clean.csv`.